# 01 - Data Download and Universe Preparation

This notebook inspects the current static S&P 500 universe and the benchmark-weight inputs used by the research pipeline. It is intentionally lightweight: core logic lives in `src/`, and this notebook only imports those functions or reads already generated outputs.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS = PROJECT_ROOT / "reports"

from src.universe import get_current_sp500_tickers
from src.data_loader import download_price_data

## Load Current S&P 500 Universe

The Version 0 universe uses the current S&P 500 constituents from Wikipedia. This is convenient for a prototype, but it is not a point-in-time historical universe.

In [2]:
tickers = get_current_sp500_tickers()
print(f"Ticker count: {len(tickers)}")
tickers[:25]

Ticker count: 503


['A',
 'AAPL',
 'ABBV',
 'ABNB',
 'ABT',
 'ACGL',
 'ACN',
 'ADBE',
 'ADI',
 'ADM',
 'ADP',
 'ADSK',
 'AEE',
 'AEP',
 'AES',
 'AFL',
 'AIG',
 'AIZ',
 'AJG',
 'AKAM',
 'ALB',
 'ALGN',
 'ALL',
 'ALLE',
 'AMAT']

## Optional Small Price Download

Set `DOWNLOAD_SAMPLE = True` to test a small `yfinance` download. It is disabled by default so the notebook can be opened without making a live internet call.

In [3]:
DOWNLOAD_SAMPLE = False

if DOWNLOAD_SAMPLE:
    sample_tickers = tickers[:5] + ["SPY"]
    sample_prices, sample_volumes = download_price_data(
        sample_tickers,
        start="2024-01-01",
        end=None,
    )
    display(sample_prices.tail())
    display(sample_volumes.tail())
else:
    print("Sample download skipped. Set DOWNLOAD_SAMPLE = True to run it.")

Sample download skipped. Set DOWNLOAD_SAMPLE = True to run it.


## Static Market-Cap Benchmark Weights

If the main pipeline has been run in `market_cap_static` mode, static benchmark weights are saved in `data/processed/static_market_cap_weights.csv`.

In [4]:
weights_path = DATA_PROCESSED / "static_market_cap_weights.csv"

if weights_path.exists():
    static_weights = pd.read_csv(weights_path, index_col=0).iloc[:, 0]
    static_weights.name = "static_market_cap_weight"
    print(f"Loaded {len(static_weights)} static market-cap weights")
    print(f"Weight sum: {static_weights.sum():.12f}")
    display(static_weights.sort_values(ascending=False).head(20).to_frame())
else:
    print(f"No static market-cap weights found at {weights_path}")

Loaded 500 static market-cap weights
Weight sum: 1.000000000000


,static_market_cap_weight
ticker,
NVDA,0.068771
GOOGL,0.066193
GOOG,0.065570
AAPL,0.057963
MSFT,0.043802
AMZN,0.041719
AVGO,0.028112
META,0.022089
TSLA,0.021015


## Data Limitations

- The current S&P 500 universe is static and introduces survivorship bias.
- Static current market-cap weights are not point-in-time and introduce look-ahead bias in historical backtests.
- `yfinance` is useful for research prototypes but is not institutional point-in-time data.
- Dual-class shares such as GOOG/GOOGL may require additional treatment before interpreting benchmark weights too literally.